In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os

In [ ]:
# Common path setup for local + Colab

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    REPO_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/masters-project")
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT.name != "masters-project" and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent

    if REPO_ROOT.name != "masters-project":
        raise RuntimeError("Could not find repo root folder named 'masters-project'.")

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
REPORTS_EVAL = REPO_ROOT / "reports" / "eval" / "modserve"
FIG_DIR = REPORTS_EVAL / "figures"
TABLE_DIR = REPORTS_EVAL / "tables"
CACHE_DIR = REPORTS_EVAL / "cache"
LOG_DIR = REPORTS_EVAL / "logs"

for p in [DATA_RAW, DATA_PROCESSED, FIG_DIR, TABLE_DIR, CACHE_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CLEAN_REQUESTS_PATH = DATA_PROCESSED / "cleaned_requests.parquet"
WINDOWS_OUTPUT_PATH = DATA_PROCESSED / "modserve_5min_windows.parquet"
BASELINE_METADATA_PATH = TABLE_DIR / "modserve_baseline_metadata.csv"

print("Repo root:", REPO_ROOT)
print("Running in Colab:", IN_COLAB)
print("Cleaned requests path:", CLEAN_REQUESTS_PATH)

In [ ]:
# load the cleaned request-level dataset
if not CLEAN_REQUESTS_PATH.exists():
    raise FileNotFoundError(
        f"Missing cleaned dataset: {CLEAN_REQUESTS_PATH}\n"
        "Run Notebook 1 first."
    )

df = pd.read_parquet(CLEAN_REQUESTS_PATH)
df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"], utc=True)

df.head()

In [ ]:
#quick sanity check
print("Rows:", len(df))
print("Time range:", df["TIMESTAMP"].min(), "to", df["TIMESTAMP"].max())
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
#create the 5-minute window column
df["window_start"] = df["TIMESTAMP"].dt.floor("5min")
df[["TIMESTAMP", "window_start"]].head()

In [ ]:
#aggregate requests into 5-minute windows
windows = (
    df.groupby("window_start")
      .agg(
          n_requests=("TIMESTAMP", "size"),
          n_text_only=("is_text_only", "sum"),
          n_image_text=("is_image_text", "sum"),
          sum_context_tokens=("ContextTokens", "sum"),
          sum_generated_tokens=("GeneratedTokens", "sum"),
          sum_images=("NumImages", "sum"),
          mean_context_tokens=("ContextTokens", "mean"),
          mean_generated_tokens=("GeneratedTokens", "mean"),
          mean_images_per_request=("NumImages", "mean"),
      )
      .reset_index()
)

windows.head()

In [ ]:
#fill in missing 5-minute intervals
full_range = pd.date_range(
    start=windows["window_start"].min(),
    end=windows["window_start"].max(),
    freq="5min",
    tz=windows["window_start"].dt.tz,
)

windows = (
    pd.DataFrame({"window_start": full_range})
      .merge(windows, on="window_start", how="left")
)

count_cols = [
    "n_requests",
    "n_text_only",
    "n_image_text",
    "sum_context_tokens",
    "sum_generated_tokens",
    "sum_images",
]

mean_cols = [
    "mean_context_tokens",
    "mean_generated_tokens",
    "mean_images_per_request",
]

windows[count_cols] = windows[count_cols].fillna(0)
windows[mean_cols] = windows[mean_cols].fillna(0)

windows.head(10)

In [ ]:
#create per-minute rate features
windows["requests_per_min"] = windows["n_requests"] / 5.0
windows["context_tokens_per_min"] = windows["sum_context_tokens"] / 5.0
windows["generated_tokens_per_min"] = windows["sum_generated_tokens"] / 5.0
windows["images_per_min"] = windows["sum_images"] / 5.0

windows["frac_image_text"] = np.where(
    windows["n_requests"] > 0,
    windows["n_image_text"] / windows["n_requests"],
    0.0,
)

windows.head()

In [ ]:
#add day column for splitting later
windows["day"] = windows["window_start"].dt.floor("D")
windows[["window_start", "day"]].head()

In [ ]:
#inspect the 5-minute dataset
print("Number of 5-minute windows:", len(windows))
print("Date range:", windows["window_start"].min(), "to", windows["window_start"].max())

display(
    windows[
        [
            "n_requests",
            "n_text_only",
            "n_image_text",
            "requests_per_min",
            "context_tokens_per_min",
            "generated_tokens_per_min",
            "images_per_min",
            "frac_image_text",
        ]
    ].describe(percentiles=[0.5, 0.9, 0.95, 0.99])
)

In [ ]:
#create chronological train / validation / test split
unique_days = sorted(windows["day"].drop_duplicates())

print("Unique days:")
for d in unique_days:
    print(d)

train_days = set(unique_days[:4])
val_days = set(unique_days[4:5])
test_days = set(unique_days[5:])

def assign_split(day):
    if day in train_days:
        return "train"
    elif day in val_days:
        return "val"
    else:
        return "test"

windows["split"] = windows["day"].apply(assign_split)
windows["split"].value_counts()

In [ ]:
# define abstract service-unit capacities
# use a temporary train subset only for estimating unit capacities

train_for_capacity = windows[windows["split"] == "train"].copy()

nonzero_image = train_for_capacity.loc[train_for_capacity["images_per_min"] > 0, "images_per_min"]
nonzero_text = train_for_capacity.loc[train_for_capacity["context_tokens_per_min"] > 0, "context_tokens_per_min"]

image_unit_capacity = max(1.0, float(nonzero_image.median())) if len(nonzero_image) else 1.0
text_unit_capacity = max(1.0, float(nonzero_text.median())) if len(nonzero_text) else 1.0

print("Image unit capacity:", image_unit_capacity)
print("Text unit capacity:", text_unit_capacity)

In [ ]:
#convert each window into required service units
windows["req_image_units"] = np.ceil(windows["images_per_min"] / image_unit_capacity).astype(int)
windows["req_text_units"] = np.ceil(windows["context_tokens_per_min"] / text_unit_capacity).astype(int)

windows["req_total_units"] = windows["req_image_units"] + windows["req_text_units"]

windows[
    [
        "window_start",
        "images_per_min",
        "context_tokens_per_min",
        "req_image_units",
        "req_text_units",
        "req_total_units",
    ]
].head(10)

In [ ]:
# define fixed provisioning levels from training data
# IMPORTANT: create train_df only after req_* columns exist

train_df = windows[windows["split"] == "train"].copy()

fixed_monolith_units = int(np.ceil(train_df["req_total_units"].quantile(0.95)))
fixed_image_units = int(np.ceil(train_df["req_image_units"].quantile(0.95)))
fixed_text_units = int(np.ceil(train_df["req_text_units"].quantile(0.95)))

print("Fixed monolith total units:", fixed_monolith_units)
print("Fixed decoupled image units:", fixed_image_units)
print("Fixed decoupled text units:", fixed_text_units)

In [ ]:
#Baseline A: Fixed Monolith
windows["prov_monolith_total_units"] = fixed_monolith_units
windows[["window_start", "req_total_units", "prov_monolith_total_units"]].head()

In [ ]:
#Baseline B: Fixed Decoupled
windows["prov_fixed_image_units"] = fixed_image_units
windows["prov_fixed_text_units"] = fixed_text_units
windows["prov_fixed_decoupled_total_units"] = (
    windows["prov_fixed_image_units"] + windows["prov_fixed_text_units"]
)

windows[
    [
        "window_start",
        "req_image_units",
        "req_text_units",
        "prov_fixed_image_units",
        "prov_fixed_text_units",
    ]
].head()

In [ ]:
#Baseline C: Reactive Decoupled
windows["prov_reactive_image_units"] = windows["req_image_units"].shift(1)
windows["prov_reactive_text_units"] = windows["req_text_units"].shift(1)

windows["prov_reactive_image_units"] = (
    windows["prov_reactive_image_units"].fillna(fixed_image_units).astype(int)
)
windows["prov_reactive_text_units"] = (
    windows["prov_reactive_text_units"].fillna(fixed_text_units).astype(int)
)

windows["prov_reactive_total_units"] = (
    windows["prov_reactive_image_units"] + windows["prov_reactive_text_units"]
)

windows[
    [
        "window_start",
        "req_image_units",
        "req_text_units",
        "prov_reactive_image_units",
        "prov_reactive_text_units",
    ]
].head(10)

In [ ]:
#compare baseline provisioning columns quickly
windows[
    [
        "window_start",
        "req_total_units",
        "prov_monolith_total_units",
        "prov_fixed_decoupled_total_units",
        "prov_reactive_total_units",
    ]
].head(15)

In [ ]:
#quick plot of required vs provisioned units over time
plot_df = windows.set_index("window_start")[
    [
        "req_total_units",
        "prov_monolith_total_units",
        "prov_fixed_decoupled_total_units",
        "prov_reactive_total_units",
    ]
].copy()

plot_df.plot(figsize=(14, 5))
plt.title("Required vs Provisioned Units Over Time")
plt.xlabel("Time")
plt.ylabel("Units")
plt.tight_layout()
plt.savefig(FIG_DIR / "required_vs_provisioned_units_over_time.png", dpi=180)
plt.show()

In [ ]:
#choose the burstiest test day
test_df = windows[windows["split"] == "test"].copy()

burst_day_table = (
    test_df.groupby("day")
           .agg(
               peak_images_per_min=("images_per_min", "max"),
               total_images=("sum_images", "sum"),
               peak_requests_per_min=("requests_per_min", "max"),
           )
           .sort_values(["peak_images_per_min", "total_images"], ascending=False)
)

burst_day_table

In [ ]:
#store the chosen burst day
burst_day = burst_day_table.index[0]
print("Chosen burst day:", burst_day)

In [ ]:
#save the 5-minute dataset with baselines
windows.to_parquet(WINDOWS_OUTPUT_PATH, index=False)
print("Saved:", WINDOWS_OUTPUT_PATH)

In [ ]:
#save metadata for later reuse
metadata = pd.DataFrame({
    "key": [
        "image_unit_capacity",
        "text_unit_capacity",
        "fixed_monolith_units",
        "fixed_image_units",
        "fixed_text_units",
        "burst_day",
    ],
    "value": [
        image_unit_capacity,
        text_unit_capacity,
        fixed_monolith_units,
        fixed_image_units,
        fixed_text_units,
        str(burst_day),
    ],
})

metadata.to_csv(BASELINE_METADATA_PATH, index=False)
print("Saved:", BASELINE_METADATA_PATH)
metadata